# 1330 버스 관측지점 선정 및 도착 Event 판정 방법

4개 노선의 공통 운행구간과 승하차 자료를 이용해 관측지점을 선정하고, 초기 실시간 수집자료(9/14~9/18)를 이용해 개별 운행 episode와 청평터미널 도착 Event 판정 방법을 설계함.


## 1. 4개 노선 공통정류장 추출

1330-2·1330-3·1330-4·1330-44의 전체 정류장 자료에서 **정류장명 기준**으로 네 노선이 모두 경유하는 공통정류장을 찾음.

In [1]:
# 4개 노선 전체 정류장 원자료 자동 다운로드
from pathlib import Path
import subprocess, sys
import pandas as pd

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

DATA_DIR = Path("/content/1330_source_data")
DATA_DIR.mkdir(exist_ok=True)

RAW_STOPS_ID = "19g1ABL6EnO-QU9LKC0I6Ud_0KINMRKJ4"
RAW_STOPS_FILE = DATA_DIR / "1330_4개노선_전체정류장.csv"

gdown.download(id=RAW_STOPS_ID, output=str(RAW_STOPS_FILE), quiet=False)

if not RAW_STOPS_FILE.exists() or RAW_STOPS_FILE.stat().st_size < 100:
    raise RuntimeError("전체 정류장 원자료 다운로드에 실패했습니다.")

try:
    stops = pd.read_csv(RAW_STOPS_FILE, encoding="utf-8-sig")
except UnicodeDecodeError:
    stops = pd.read_csv(RAW_STOPS_FILE, encoding="cp949")

stops.columns = stops.columns.astype(str).str.strip()

print("전체 정류장 데이터 크기:", stops.shape)
print("컬럼:", stops.columns.tolist())
display(stops.head())


Downloading...
From: https://drive.google.com/uc?id=19g1ABL6EnO-QU9LKC0I6Ud_0KINMRKJ4
To: /content/1330_source_data/1330_4개노선_전체정류장.csv
100%|██████████| 56.5k/56.5k [00:00<00:00, 16.9MB/s]

전체 정류장 데이터 크기: (677, 8)
컬럼: ['route_name', 'route_id', 'station_id', 'station_name', 'mobile_no', 'station_seq', 'x', 'y']


,route_name,route_id,station_id,station_name,mobile_no,station_seq,x,y
0,1330-2,239000110,239000282,가평터미널,44613.0,1,127.515333,37.824633
1,1330-2,239000110,277102405,가평터미널(경유),NaN,2,127.514567,37.822567
2,1330-2,239000110,239000335,남이오거리,44146.0,3,127.512867,37.820133
3,1330-2,239000110,277102704,하색리(경유),NaN,4,127.497567,37.810250
4,1330-2,239000110,239000235,두밀리입구,44125.0,5,127.487017,37.803750


In [2]:
# 1330-2·1330-3·1330-4·1330-44 공통정류장 추출
# 공통정류장 탐색 단계에서는 방향별 station_id가 달라도
# 같은 정류장명이라면 같은 공통 경유지로 본다.

target_routes = ["1330-2", "1330-3", "1330-4", "1330-44"]

route_candidates = ["route_name", "노선명", "노선번호", "ROUTE_NM"]
station_id_candidates = ["station_id", "정류소ID", "정류장ID", "STATION_ID"]
station_name_candidates = ["station_name", "정류소명", "정류장명", "STATION_NM"]

route_col = next((c for c in route_candidates if c in stops.columns), None)
station_id_col = next((c for c in station_id_candidates if c in stops.columns), None)
station_name_col = next((c for c in station_name_candidates if c in stops.columns), None)

if not all([route_col, station_id_col, station_name_col]):
    raise KeyError(
        "공통정류장 추출에 필요한 컬럼을 찾지 못했습니다. "
        f"현재 컬럼: {stops.columns.tolist()}"
    )

target = stops[stops[route_col].astype(str).isin(target_routes)].copy()

# 1) 정류장명별로 몇 개 노선이 경유하는지 확인
common_names = (
    target.groupby(station_name_col, dropna=False)[route_col]
    .nunique()
)
common_names = common_names[common_names == 4].index

# 2) 공통 정류장명에 해당하는 실제 정류장 ID들을 모두 보존
#    (방향별 ID가 다를 수 있으므로 이 단계에서 하나로 합치지 않음)
common4 = (
    target[target[station_name_col].isin(common_names)]
    [[station_id_col, station_name_col, route_col]]
    .drop_duplicates()
    .rename(columns={
        station_id_col: "station_id",
        station_name_col: "station_name",
        route_col: "route_name"
    })
    .sort_values(["station_name", "route_name", "station_id"])
    .reset_index(drop=True)
)

print("4개 노선 공통 정류장명 수:", common4["station_name"].nunique())
print("방향별 station_id를 포함한 행 수:", len(common4))
display(common4)


4개 노선 공통 정류장명 수: 81
방향별 station_id를 포함한 행 수: 508


,station_id,station_name,route_name
0,222001318,가운휴먼시아.다산효성해링턴타워.새마을금고다산본점(중),1330-2
1,222001319,가운휴먼시아.다산효성해링턴타워.새마을금고다산본점(중),1330-2
2,222001318,가운휴먼시아.다산효성해링턴타워.새마을금고다산본점(중),1330-3
3,222001319,가운휴먼시아.다산효성해링턴타워.새마을금고다산본점(중),1330-3
4,222001318,가운휴먼시아.다산효성해링턴타워.새마을금고다산본점(중),1330-4
...,...,...,...
503,222000030,화도낚시터.돌고개,1330-44
504,222000188,효성해링턴플레이스.구룡터,1330-2
505,222000188,효성해링턴플레이스.구룡터,1330-3
506,222000188,효성해링턴플레이스.구룡터,1330-4


## 2. 4~6월 승하차 원자료 불러오기

공통정류장 후보의 실제 이용량을 비교하기 위해 월별 승하차 4~6월 자료를 통합함.


In [3]:
# 4~6월 승하차 원자료 자동 다운로드
import zipfile
import os

MONTHLY_FILES = {
    "04": ("1Gmef6wHbNh2c0IVB7FwhRyX8puAgvzeR", DATA_DIR / "2026_04_D-45.zip"),
    "05": ("1Ie3_qStREXFlvzU7g_lcN4C54x5Ryrzo", DATA_DIR / "2026_05_D-45.zip"),
    "06": ("1xVpk6QoJxaRRa2HdsyIXU8hSTkd87KYC", DATA_DIR / "2026_06_D-45.zip"),
}

for month, (file_id, output_path) in MONTHLY_FILES.items():
    gdown.download(id=file_id, output=str(output_path), quiet=False)

    if not output_path.exists() or output_path.stat().st_size < 100:
        raise RuntimeError(f"{month}월 승하차 원자료 다운로드에 실패했습니다.")

    print("\nZIP:", output_path.name)
    with zipfile.ZipFile(output_path) as z:
        print("내부 파일:", z.namelist()[:10])


Downloading...
From: https://drive.google.com/uc?id=1Gmef6wHbNh2c0IVB7FwhRyX8puAgvzeR
To: /content/1330_source_data/2026_04_D-45.zip
100%|██████████| 15.2M/15.2M [00:00<00:00, 80.9MB/s]



ZIP: 2026_04_D-45.zip
내부 파일: ['CARD_STATION45_202604.tar']


Downloading...
From: https://drive.google.com/uc?id=1Ie3_qStREXFlvzU7g_lcN4C54x5Ryrzo
To: /content/1330_source_data/2026_05_D-45.zip
100%|██████████| 15.6M/15.6M [00:00<00:00, 58.2MB/s]



ZIP: 2026_05_D-45.zip
내부 파일: ['CARD_STATION45_202605.tar']


Downloading...
From: https://drive.google.com/uc?id=1xVpk6QoJxaRRa2HdsyIXU8hSTkd87KYC
To: /content/1330_source_data/2026_06_D-45.zip
100%|██████████| 15.2M/15.2M [00:00<00:00, 74.3MB/s]


ZIP: 2026_06_D-45.zip
내부 파일: ['CARD_STATION45_202606.tar']


In [4]:
import zipfile
import tarfile
import io

for _, output_path in MONTHLY_FILES.values():
    print("\nZIP:", output_path.name)

    with zipfile.ZipFile(output_path) as z:
        tar_name = z.namelist()[0]
        tar_bytes = z.read(tar_name)

        with tarfile.open(fileobj=io.BytesIO(tar_bytes)) as tar:
            print("TAR 내부:", tar.getnames()[:10])



ZIP: 2026_04_D-45.zip
TAR 내부: ['CARD_STATION45_20260401.csv.gz', 'CARD_STATION45_20260402.csv.gz', 'CARD_STATION45_20260403.csv.gz', 'CARD_STATION45_20260404.csv.gz', 'CARD_STATION45_20260405.csv.gz', 'CARD_STATION45_20260406.csv.gz', 'CARD_STATION45_20260407.csv.gz', 'CARD_STATION45_20260408.csv.gz', 'CARD_STATION45_20260409.csv.gz', 'CARD_STATION45_20260410.csv.gz']

ZIP: 2026_05_D-45.zip
TAR 내부: ['CARD_STATION45_20260501.csv.gz', 'CARD_STATION45_20260502.csv.gz', 'CARD_STATION45_20260503.csv.gz', 'CARD_STATION45_20260504.csv.gz', 'CARD_STATION45_20260505.csv.gz', 'CARD_STATION45_20260506.csv.gz', 'CARD_STATION45_20260507.csv.gz', 'CARD_STATION45_20260508.csv.gz', 'CARD_STATION45_20260509.csv.gz', 'CARD_STATION45_20260510.csv.gz']

ZIP: 2026_06_D-45.zip
TAR 내부: ['CARD_STATION45_20260601.csv.gz', 'CARD_STATION45_20260602.csv.gz', 'CARD_STATION45_20260603.csv.gz', 'CARD_STATION45_20260604.csv.gz', 'CARD_STATION45_20260605.csv.gz', 'CARD_STATION45_20260606.csv.gz', 'CARD_STATION45_2026

In [5]:
# 4~6월 승하차 원본데이터 통합
import pandas as pd
import zipfile
import tarfile
import gzip
import io

monthly_data = []

for month, (_, zip_path) in MONTHLY_FILES.items():
    print("불러오는 중:", zip_path.name)

    with zipfile.ZipFile(zip_path) as z:
        tar_name = z.namelist()[0]
        tar_bytes = z.read(tar_name)

        with tarfile.open(fileobj=io.BytesIO(tar_bytes)) as tar:
            for member in tar.getmembers():
                if member.name.endswith(".csv.gz"):
                    gz_file = tar.extractfile(member)

                    with gzip.open(gz_file, "rt", encoding="cp949") as f:
                        temp = pd.read_csv(f, low_memory=False)
                        monthly_data.append(temp)

ridership = pd.concat(monthly_data, ignore_index=True)

print("\n불러오기 완료!")
print("전체 데이터 크기:", ridership.shape)
display(ridership.head())


불러오는 중: 2026_04_D-45.zip
불러오는 중: 2026_05_D-45.zip
불러오는 중: 2026_06_D-45.zip

불러오기 완료!
전체 데이터 크기: (2913546, 9)


,승하차일자,관할관청,정류소ID,정류소번호,정류소명,승차합계,초승,환승,하차
0,20260401,가평군,239000171,44498,2반카페거리,2,2,0,6
1,20260401,가평군,239000747,44737,4반펜션촌,0,0,0,1
2,20260401,가평군,239000748,44738,4반펜션촌,4,4,0,0
3,20260401,가평군,239000285,44165,가골,1,1,0,0
4,20260401,가평군,239000437,44166,가골,0,0,0,3


## 3. 가평권·마석권 후보 정류장 이용량 비교

공통정류장 가운데 가평권과 마석권 후보의 승하차 규모를 비교하여 실시간 관측지점 후보를 좁힘.

In [6]:
# ==========================================
# 4. 가평권 / 마석권 공통정류장 이용량 비교
# ==========================================

# 공통정류장 추출 셀이 먼저 실행되었는지 확인
if "common4" not in globals():
    raise RuntimeError("위의 '4개 노선 공통정류장 추출' 셀을 먼저 실행해주세요.")

# 정류소 ID 자료형 통일
ridership["정류소ID"] = ridership["정류소ID"].astype(str)
common4["station_id"] = common4["station_id"].astype(str)

# 날짜 형식 변환
ridership["승하차일자"] = pd.to_datetime(
    ridership["승하차일자"].astype(str),
    format="%Y%m%d"
)

# ------------------------------------------
# ① 가평권 후보
# ------------------------------------------

gapyeong_ids = set(
    ridership.loc[
        ridership["관할관청"] == "가평군",
        "정류소ID"
    ]
)

gapyeong_common = common4[
    common4["station_id"].isin(gapyeong_ids)
].copy()


# ------------------------------------------
# ② 마석권 후보
# ------------------------------------------

maseok_keywords = [
    "마석",
    "모란공원",
    "지현마을",
    "남양주세무서",
    "천마산",
    "화도"
]

maseok_common = common4[
    common4["station_name"].astype(str).str.contains(
        "|".join(maseok_keywords),
        na=False
    )
].copy()


# ------------------------------------------
# ③ 정류장별 3개월 일평균 승차량 계산 함수
# ------------------------------------------

def make_ranking(candidate_df):

    target_ids = set(candidate_df["station_id"])

    temp = ridership[
        ridership["정류소ID"].isin(target_ids)
    ].copy()

    daily = (
        temp
        .groupby(
            ["승하차일자", "정류소ID", "정류소명"],
            as_index=False
        )["승차합계"]
        .sum()
    )

    ranking = (
        daily
        .groupby(["정류소ID", "정류소명"])
        .agg(
            일평균승차=("승차합계", "mean"),
            총승차=("승차합계", "sum"),
            관측일수=("승하차일자", "nunique")
        )
        .reset_index()
        .sort_values("일평균승차", ascending=False)
    )

    return ranking


gapyeong_ranking = make_ranking(gapyeong_common)
maseok_ranking = make_ranking(maseok_common)


print("===== 가평군 4개 노선 공통정류장 =====")
display(gapyeong_ranking)

print("\n===== 마석권 4개 노선 공통정류장 =====")
display(maseok_ranking)

===== 가평군 4개 노선 공통정류장 =====


,정류소ID,정류소명,일평균승차,총승차,관측일수
27,239000738,청평터미널,216.901099,19738,91
16,239000225,청평농협,108.560440,9879,91
6,239000215,대성리역,74.494505,6779,91
15,239000224,시장입구,60.032967,5463,91
11,239000220,원대성리,28.230769,2569,91
8,239000217,대성리MT장,27.098901,2466,91
9,239000218,대성리역,21.615385,1967,91
17,239000226,청평중학교,21.307692,1939,91
0,239000182,검문소,14.560440,1325,91
2,239000211,삼거리.시장,13.494505,1228,91



===== 마석권 4개 노선 공통정류장 =====


,정류소ID,정류소명,일평균승차,총승차,관측일수
0,222000010,마석역,1645.824176,149770,91
12,222000742,마석역,1082.978022,98551,91
6,222000190,천마산휴게소.한누리요양병원,578.000000,52598,91
8,222000192,남양주세무서.천마산수련장,341.120879,31042,91
2,222000015,지현마을.남양주고등학교.화도동부출장소,258.582418,23531,91
1,222000014,모란공원.마석그랜드힐아파트,248.461538,22610,91
11,222000199,천마산휴게소.한누리요양병원,231.032967,21024,91
7,222000191,마석가구공단.천마산스키장입구,143.032967,13016,91
9,222000196,남양주세무서.천마산수련장,116.890110,10637,91
10,222000198,마석가구공단.천마산스키장입구,104.340659,9495,91


## 4. 서울방향 정류장 확인 및 관측지점 선정

후보 정류장의 `station_id`와 노선 내 순서인 `station_seq`를 확인하여 서울방향 정류장을 구분함. 공통 운행구간과 이용량을 함께 고려해 청평터미널과 마석역을 관측지점으로 선정함.


In [7]:
# ==========================================
# 5. 가평권 / 마석권 후보 정류장의 방향 확인
# ==========================================

# 승차량 상위 후보 몇 곳
check_station_ids = [
    # 가평권
    "239000738",   # 청평터미널
    "239000225",   # 청평농협
    "239000215",   # 대성리역
    "239000218",   # 대성리역

    # 마석권
    "222000010",   # 마석역
    "222000742"    # 마석역
]

direction_check = (
    stops[
        stops["station_id"].astype(str).isin(check_station_ids)
        & stops["route_name"].isin(["1330-2", "1330-3", "1330-4", "1330-44"])
    ]
    [
        ["route_name", "station_id", "station_name", "station_seq"]
    ]
    .sort_values(["station_name", "route_name", "station_seq"])
)

display(direction_check)

,route_name,station_id,station_name,station_seq
24,1330-2,239000218,대성리역,25
135,1330-2,239000215,대성리역,136
190,1330-3,239000218,대성리역,34
301,1330-3,239000215,대성리역,145
357,1330-4,239000218,대성리역,26
467,1330-4,239000215,대성리역,136
529,1330-44,239000218,대성리역,36
640,1330-44,239000215,대성리역,147
35,1330-2,222000010,마석역,36
124,1330-2,222000742,마석역,125


## 5. 초기 실시간 수집자료 불러오기

청평터미널·마석역의 9/14~9/18 실시간 API 수집자료를 불러오고 분석 변수를 정리함.


In [8]:
# ============================================================
# 평가용 자동 데이터 로더
# Google Drive 공개 링크 → xlsx 자동 다운로드 → 분석 시작
# ============================================================

from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

FILE_ID = "1cQLU8PwubCz9YBD_HrxQB4Urdo7My-X6"
DOWNLOAD_URL = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
LOCAL_FILE = Path("/content/1330_preprocessing_input.xlsx")

def download_input():
    try:
        urllib.request.urlretrieve(DOWNLOAD_URL, LOCAL_FILE)
    except Exception as e:
        raise RuntimeError(
            "공개 데이터 자동 다운로드에 실패했습니다. "
            "Google Drive 공유 설정이 '링크가 있는 모든 사용자 - 뷰어'인지 확인해주세요."
        ) from e

    if not LOCAL_FILE.exists() or LOCAL_FILE.stat().st_size < 1000:
        raise RuntimeError(
            "다운로드된 파일이 정상적인 xlsx로 보이지 않습니다. "
            "Google Drive 공개 권한을 확인해주세요."
        )

    # HTML 로그인/권한 페이지가 내려온 경우 즉시 잡아냄
    with open(LOCAL_FILE, "rb") as f:
        head = f.read(200).lower()
    if b"<html" in head or b"<!doctype" in head:
        raise RuntimeError(
            "xlsx 대신 Google 로그인/권한 페이지가 내려왔습니다. "
            "공유 설정을 '링크가 있는 모든 사용자 - 뷰어'로 바꿔주세요."
        )

    print("데이터 자동 다운로드 완료:", LOCAL_FILE)
    print("파일 크기:", round(LOCAL_FILE.stat().st_size / 1024, 1), "KB")
    return LOCAL_FILE

input_file = download_input()

live_file = input_file
live = pd.read_excel(live_file)
live["recorded_at"] = pd.to_datetime(live["recorded_at"])

required_cols = {
    "recorded_at", "station_name", "route_name",
    "vehicle_id", "predict_time_sec", "location_no"
}
missing = required_cols - set(live.columns)
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing)}")

for col in ["predict_time_sec", "location_no"]:
    live[col] = pd.to_numeric(live[col], errors="coerce")

live["date"] = live["recorded_at"].dt.date

print("행 수:", len(live))
print("수집 기간:", live["recorded_at"].min(), "~", live["recorded_at"].max())
print("정류장:", sorted(live["station_name"].dropna().unique()))


데이터 자동 다운로드 완료: /content/1330_preprocessing_input.xlsx
파일 크기: 394.9 KB
행 수: 6581
수집 기간: 2026-09-14 19:16:48 ~ 2026-09-18 20:58:49
정류장: ['마석역', '청평터미널']


## 6. 운행 Episode 생성

동일 차량의 반복 운행을 구분하기 위해 동일 정류장·날짜·노선·차량에서 관측 간격이 90분 이상이면 새로운 episode로 구분함.


In [9]:
# ============================================================
# 2. 같은 차량의 운행을 episode로 나누기
# ============================================================

live = live.sort_values(
    ["station_name", "date", "route_name", "vehicle_id", "recorded_at"]
).reset_index(drop=True)

live["time_gap_min"] = (
    live.groupby(["station_name", "date", "route_name", "vehicle_id"])["recorded_at"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# 같은 차량이라도 90분 이상 뒤 다시 등장하면 다른 운행으로 처리
live["new_episode"] = (
    live["time_gap_min"].isna()
    | (live["time_gap_min"] >= 90)
)

live["episode_no"] = (
    live.groupby(["station_name", "date", "route_name", "vehicle_id"])["new_episode"]
    .cumsum()
)

live["episode_id"] = (
    live["date"].astype(str)
    + "_" + live["station_name"].astype(str)
    + "_" + live["route_name"].astype(str)
    + "_" + live["vehicle_id"].astype(str)
    + "_" + live["episode_no"].astype(str)
)

# 날짜·정류장별 마지막 수집시각
daily_collection_end = (
    live.groupby(["date", "station_name"])["recorded_at"]
    .max()
    .rename("collection_end")
    .reset_index()
)

df14 = live.merge(
    daily_collection_end,
    on=["date", "station_name"],
    how="left"
)

print("===== episode 생성 결과 =====")
print(df14.groupby("station_name")["episode_id"].nunique())


===== episode 생성 결과 =====
station_name
마석역      193
청평터미널    191
Name: episode_id, dtype: int64


## 7. ETA·위치 단일 기준의 한계 확인

약 2분 간격의 관측에서는 특정 ETA 또는 `location_no` 순간을 놓칠 수 있으므로 여러 단일 기준을 비교함.


In [10]:
# ============================================================
# 3. 단일 ETA/location 기준만으로 도착을 판정할 수 있는지 확인
# ============================================================

episode_summary = (
    df14.groupby(
        ["station_name", "date", "route_name", "vehicle_id", "episode_id"],
        as_index=False
    )
    .agg(
        first_seen=("recorded_at", "min"),
        last_seen=("recorded_at", "max"),
        n_obs=("recorded_at", "size"),
        min_eta=("predict_time_sec", "min"),
        min_location=("location_no", "min"),
        collection_end=("collection_end", "first")
    )
)

episode_summary["loc1"] = episode_summary["min_location"] <= 1
episode_summary["eta60"] = episode_summary["min_eta"] <= 60
episode_summary["eta180"] = episode_summary["min_eta"] <= 180
episode_summary["loc1_or_eta60"] = (
    episode_summary["loc1"] | episode_summary["eta60"]
)
episode_summary["loc2_eta180"] = (
    (episode_summary["min_location"] <= 2)
    & (episode_summary["min_eta"] <= 180)
)

candidate_cols = [
    "loc1", "eta60", "eta180",
    "loc1_or_eta60", "loc2_eta180"
]

candidate_result = pd.DataFrame({
    "후보기준": candidate_cols,
    "포착 episode 수": [
        int(episode_summary[c].sum()) for c in candidate_cols
    ]
})

candidate_result["전체 episode 수"] = len(episode_summary)
candidate_result["포착 비율(%)"] = (
    candidate_result["포착 episode 수"]
    / candidate_result["전체 episode 수"] * 100
).round(1)

display(candidate_result)

candidate_sum = episode_summary[candidate_cols].astype(int).sum(axis=1)
disagreement = (
    (candidate_sum > 0)
    & (candidate_sum < len(candidate_cols))
).sum()

print("후보 기준별 판단이 서로 다른 episode:", int(disagreement))


,후보기준,포착 episode 수,전체 episode 수,포착 비율(%)
0,loc1,309,384,80.5
1,eta60,118,384,30.7
2,eta180,335,384,87.2
3,loc1_or_eta60,309,384,80.5
4,loc2_eta180,334,384,87.0


후보 기준별 판단이 서로 다른 episode: 225


## 8. 단순 청평–마석 매칭의 문제 확인

동일 날짜·차량만으로 연결했을 때 서로 다른 회차가 연결되어 비정상적으로 긴 이동시간이 발생하는지 확인함.


In [11]:
# ============================================================
# 4. 단순 청평→마석 매칭을 해보면 왜 문제가 생기는지 확인
# ============================================================

# 먼저 episode별 첫/마지막 관측시각을 요약
episode_for_match = (
    df14.sort_values("recorded_at")
    .groupby(
        ["date", "station_name", "route_name", "vehicle_id", "episode_id"],
        as_index=False
    )
    .agg(
        first_seen=("recorded_at", "min"),
        last_seen=("recorded_at", "max")
    )
)

cp_naive = (
    episode_for_match[episode_for_match["station_name"] == "청평터미널"]
    .rename(columns={
        "episode_id": "cp_episode",
        "last_seen": "cp_last_seen"
    })
)

ms_naive = (
    episode_for_match[episode_for_match["station_name"] == "마석역"]
    .rename(columns={
        "episode_id": "ms_episode",
        "last_seen": "ms_last_seen"
    })
)

# '같은 날짜 + 같은 차량'만으로 연결하면 같은 날의 다른 회차까지 붙을 수 있음
naive_pairs = cp_naive.merge(
    ms_naive[["date", "vehicle_id", "ms_episode", "ms_last_seen"]],
    on=["date", "vehicle_id"],
    how="inner"
)

naive_pairs["travel_min"] = (
    naive_pairs["ms_last_seen"] - naive_pairs["cp_last_seen"]
).dt.total_seconds() / 60

positive_naive = naive_pairs[naive_pairs["travel_min"] > 0].copy()

print("단순 매칭 후보 수:", len(positive_naive))
if len(positive_naive):
    print("단순 매칭 이동시간 중앙값:",
          round(positive_naive["travel_min"].median(), 1), "분")
    print("단순 매칭 최대 이동시간:",
          round(positive_naive["travel_min"].max(), 1), "분")
    print("60분 초과 비정상 후보:",
          int((positive_naive["travel_min"] > 60).sum()), "건")

    print("\n이동시간이 가장 긴 사례 5개")
    display(
        positive_naive.nlargest(5, "travel_min")[
            ["date", "vehicle_id", "cp_last_seen", "ms_last_seen", "travel_min"]
        ]
    )


단순 매칭 후보 수: 382
단순 매칭 이동시간 중앙값: 160.0 분
단순 매칭 최대 이동시간: 896.0 분
60분 초과 비정상 후보: 191 건

이동시간이 가장 긴 사례 5개


,date,vehicle_id,cp_last_seen,ms_last_seen,travel_min
542,2026-09-18,239000007,2026-09-18 06:02:47,2026-09-18 20:58:49,896.033333
280,2026-09-16,239000026,2026-09-16 06:02:48,2026-09-16 20:58:48,896.000000
409,2026-09-17,239000017,2026-09-17 06:04:47,2026-09-17 20:58:48,894.016667
120,2026-09-15,239000007,2026-09-15 06:04:48,2026-09-15 20:58:48,894.000000
18,2026-09-15,239000047,2026-09-15 06:16:47,2026-09-15 20:58:48,882.016667


## 9. 청평–마석 이동시간 분포 탐색

동일 날짜·노선·차량에서 청평 이후 가장 가까운 마석 episode를 연결해 이동시간 분포를 확인함.


In [12]:
# ============================================================
# 5. 시간범위를 정하기 전: 청평→마석 이동시간 분포 탐색
# ============================================================
# 목적
# 1) 15~60분이라는 범위를 먼저 강제로 넣지 않는다.
# 2) 같은 날짜 + 같은 노선 + 같은 차량 중,
#    각 청평 episode 이후 '가장 가까운' 마석 episode를 먼저 연결한다.
# 3) 그 결과의 분포를 보고 대표 이동시간과 넓은 후보 window의 근거를 확인한다.

# 초기 단계에서 비교했던 후보 기준 중 하나인
# location_no <= 2 AND ETA <= 180초 관측을 사용
explore_rows = df14[
    (df14["location_no"] <= 2)
    & (df14["predict_time_sec"] <= 180)
].copy()

explore_events = (
    explore_rows.sort_values("recorded_at")
    .groupby(
        ["date", "station_name", "route_name", "vehicle_id", "episode_id"],
        as_index=False
    )
    .last()
)

cp_explore = (
    explore_events[explore_events["station_name"] == "청평터미널"]
    [["date", "route_name", "vehicle_id", "episode_id", "recorded_at"]]
    .rename(columns={
        "episode_id": "cp_episode",
        "recorded_at": "cp_time"
    })
)

ms_explore = (
    explore_events[explore_events["station_name"] == "마석역"]
    [["date", "route_name", "vehicle_id", "episode_id", "recorded_at"]]
    .rename(columns={
        "episode_id": "ms_episode",
        "recorded_at": "ms_time"
    })
)

# 같은 날짜 + 같은 노선 + 같은 차량의 가능한 조합
explore_pairs = cp_explore.merge(
    ms_explore,
    on=["date", "route_name", "vehicle_id"],
    how="inner"
)

explore_pairs["travel_min"] = (
    explore_pairs["ms_time"] - explore_pairs["cp_time"]
).dt.total_seconds() / 60

# 청평 이후 마석에 나타난 후보만 남김
forward_explore = explore_pairs[
    explore_pairs["travel_min"] > 0
].copy()

# 아직 15~60분 같은 범위를 쓰지 않고,
# 각 청평 episode 이후 가장 가까운 마석 episode를 선택
nearest_explore = (
    forward_explore
    .sort_values("travel_min")
    .drop_duplicates(subset=["cp_episode"], keep="first")
    .copy()
)

print("===== 시간범위 적용 전 이동시간 탐색 =====")
print("연결된 청평 episode:", len(nearest_explore))
print(nearest_explore["travel_min"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
))

print("\n===== 이동시간 구간 =====")
print("<15분:", int((nearest_explore["travel_min"] < 15).sum()))
print("15~20분:", int(nearest_explore["travel_min"].between(15, 20, inclusive="left").sum()))
print("20~35분:", int(nearest_explore["travel_min"].between(20, 35, inclusive="both").sum()))
print("35~60분:", int(((nearest_explore["travel_min"] > 35) & (nearest_explore["travel_min"] <= 60)).sum()))
print(">60분:", int((nearest_explore["travel_min"] > 60).sum()))

# 이 탐색에서 나온 중앙값을 이후 복수 후보 tie-break의 대표값으로 사용
representative_travel_min = nearest_explore["travel_min"].median()
print("\n탐색 단계 중앙값:", round(representative_travel_min, 1), "분")


===== 시간범위 적용 전 이동시간 탐색 =====
연결된 청평 episode: 154
count    154.000000
mean      33.156602
std       48.074970
min       19.983333
5%        20.016667
25%       24.000000
50%       26.000000
75%       28.000000
95%       30.033333
max      332.000000
Name: travel_min, dtype: float64

===== 이동시간 구간 =====
<15분: 0
15~20분: 1
20~35분: 149
35~60분: 0
>60분: 4

탐색 단계 중앙값: 26.0 분


## 10. 청평–마석 Episode 1:1 매칭

15~60분 범위의 후보를 사용하고, 복수 후보가 있을 때 관측 분포의 대표값인 약 26분에 가까운 조합을 우선하여 1:1 매칭함.


In [13]:
# ============================================================
# 6. 개선된 청평 episode ↔ 마석 episode 1:1 매칭
# ============================================================

episode = (
    df14.sort_values("recorded_at")
    .groupby(
        ["date", "station_name", "route_name", "vehicle_id", "episode_id"],
        as_index=False
    )
    .agg(
        first_seen=("recorded_at", "min"),
        last_seen=("recorded_at", "max"),
        n_obs=("recorded_at", "size"),
        first_eta=("predict_time_sec", "first"),
        last_eta=("predict_time_sec", "last"),
        first_location=("location_no", "first"),
        last_location=("location_no", "last"),
        collection_end=("collection_end", "first")
    )
)

cp17 = (
    episode[episode["station_name"] == "청평터미널"]
    .copy()
    .rename(columns={
        "episode_id": "cp_episode",
        "first_seen": "cp_first_seen",
        "last_seen": "cp_last_seen",
        "n_obs": "cp_n_obs",
        "first_eta": "cp_first_eta",
        "last_eta": "cp_last_eta",
        "first_location": "cp_first_location",
        "last_location": "cp_last_location",
        "collection_end": "cp_collection_end"
    })
)

ms17 = (
    episode[episode["station_name"] == "마석역"]
    .copy()
    .rename(columns={
        "episode_id": "ms_episode",
        "first_seen": "ms_first_seen",
        "last_seen": "ms_last_seen",
        "n_obs": "ms_n_obs",
        "first_eta": "ms_first_eta",
        "last_eta": "ms_last_eta",
        "first_location": "ms_first_location",
        "last_location": "ms_last_location",
        "collection_end": "ms_collection_end"
    })
)

pairs17 = cp17.merge(
    ms17,
    on=["date", "route_name", "vehicle_id"],
    how="inner"
)

pairs17["last_seen_travel_min"] = (
    pairs17["ms_last_seen"] - pairs17["cp_last_seen"]
).dt.total_seconds() / 60

# 앞 탐색에서 정상 관측은 주로 20~35분에 모였고,
# 60분 초과는 수시간 뒤 다른 회차가 연결된 사례였다.
# 따라서 정상 범위보다 넓은 15~60분을 '후보 window'로 사용한다.
possible17 = pairs17[
    pairs17["last_seen_travel_min"].between(15, 60, inclusive="both")
].copy()

# 복수 후보가 있을 때만 앞 탐색에서 계산한 중앙값(약 26분)에
# 가장 가까운 후보를 tie-break로 선택한다.
possible17["distance_from_representative"] = (
    possible17["last_seen_travel_min"] - representative_travel_min
).abs()

matched17 = (
    possible17
    .sort_values(
        ["date", "route_name", "vehicle_id", "cp_episode", "distance_from_representative"]
    )
    .drop_duplicates(subset=["cp_episode"], keep="first")
    .sort_values("distance_from_representative")
    .drop_duplicates(subset=["ms_episode"], keep="first")
    .copy()
)

print("청평 episode:", len(cp17))
print("마석 episode:", len(ms17))
print("15~60분 연결 후보:", len(possible17))
print("최종 1:1 매칭:", len(matched17))

if len(matched17):
    print(
        "매칭된 청평→마석 관측시간 차이 중앙값:",
        round(matched17["last_seen_travel_min"].median(), 1), "분"
    )


청평 episode: 191
마석 episode: 193
15~60분 연결 후보: 183
최종 1:1 매칭: 183
매칭된 청평→마석 관측시간 차이 중앙값: 26.0 분


## 11. 매칭 결과 점검

최종 매칭의 이동시간 분포를 확인하여 다른 회차가 잘못 연결된 사례가 남아 있는지 점검함.


In [14]:
# ============================================================
# 7. 개선된 매칭 결과가 상식적인 범위인지 확인
# ============================================================

if len(matched17):
    matched_20_35 = matched17[
        matched17["last_seen_travel_min"].between(20, 35, inclusive="both")
    ]

    print("최종 1:1 매칭:", len(matched17), "건")
    print("20~35분 범위:", len(matched_20_35), "건")
    print("20~35분 비율:",
          round(len(matched_20_35) / len(matched17) * 100, 1), "%")
    print("이동시간 중앙값:",
          round(matched17["last_seen_travel_min"].median(), 1), "분")
    print("최소~최대:",
          round(matched17["last_seen_travel_min"].min(), 1), "~",
          round(matched17["last_seen_travel_min"].max(), 1), "분")


최종 1:1 매칭: 183 건
20~35분 범위: 181 건
20~35분 비율: 98.9 %
이동시간 중앙값: 26.0 분
최소~최대: 20.0 ~ 38.0 분


## 12. 수집 시작·종료 경계 확인

수집 시작 직후 이미 정류장 가까이에 있던 차량과 수집 종료 직전에 기록이 끊긴 episode를 구분함.


In [15]:
# ============================================================
# 8. 수집 시작·종료 경계가 실제로 존재하는지 확인
# ============================================================

# 청평 episode별 접근 흐름을 먼저 계산
def check_approach_flow(group):
    g = group.sort_values("recorded_at").copy()

    eta = pd.to_numeric(g["predict_time_sec"], errors="coerce").dropna()
    loc = pd.to_numeric(g["location_no"], errors="coerce").dropna()

    min_eta = eta.min() if len(eta) else np.nan
    min_location = loc.min() if len(loc) else np.nan

    eta_approached = len(eta) >= 2 and eta.iloc[-1] < eta.iloc[0]
    location_approached = len(loc) >= 2 and loc.iloc[-1] < loc.iloc[0]

    came_near = (
        (pd.notna(min_location) and min_location <= 3)
        or (pd.notna(min_eta) and min_eta <= 300)
    )

    return pd.Series({
        "min_eta_episode": min_eta,
        "min_location_episode": min_location,
        "approach_flow": came_near and (eta_approached or location_approached)
    })

cp_flow = (
    df14[df14["station_name"] == "청평터미널"]
    .groupby(["date", "route_name", "vehicle_id", "episode_id"])
    .apply(check_approach_flow, include_groups=False)
    .reset_index()
    .rename(columns={"episode_id": "cp_episode"})
)

boundary_check = cp17.merge(
    cp_flow,
    on=["date", "route_name", "vehicle_id", "cp_episode"],
    how="left"
)

boundary_check = boundary_check.merge(
    matched17[["cp_episode", "ms_episode"]],
    on="cp_episode",
    how="left"
)

boundary_check["matched_to_maseok"] = boundary_check["ms_episode"].notna()

# 1) 수집 종료 직전 끊긴 episode
boundary_check["minutes_before_collection_end"] = (
    boundary_check["cp_collection_end"] - boundary_check["cp_last_seen"]
).dt.total_seconds() / 60

boundary_check["cut_by_collection_end"] = (
    boundary_check["minutes_before_collection_end"]
    .between(0, 10, inclusive="both")
)

# 2) 수집 시작 직후 이미 청평 가까이에 있던 episode
boundary_check["start_boundary_arrival"] = (
    (boundary_check["cp_first_seen"].dt.hour == 6)
    & (boundary_check["cp_first_seen"].dt.minute <= 2)
    & (boundary_check["min_location_episode"] <= 3)
    & (boundary_check["min_eta_episode"] <= 300)
    & boundary_check["matched_to_maseok"].fillna(False)
)

print("===== 수집 경계 확인 =====")
print("수집 종료 10분 이내에 끊긴 청평 episode:",
      int(boundary_check["cut_by_collection_end"].sum()))
print("수집 시작 경계 예외에 해당하는 episode:",
      int(boundary_check["start_boundary_arrival"].sum()))

print("\n[수집 종료 경계 예시]")
display(
    boundary_check.loc[
        boundary_check["cut_by_collection_end"],
        ["date", "route_name", "vehicle_id", "cp_episode",
         "cp_last_seen", "cp_collection_end", "minutes_before_collection_end"]
    ].head(10)
)

print("\n[수집 시작 경계 예시]")
display(
    boundary_check.loc[
        boundary_check["start_boundary_arrival"],
        ["date", "route_name", "vehicle_id", "cp_episode",
         "cp_first_seen", "min_location_episode", "min_eta_episode",
         "matched_to_maseok"]
    ].head(10)
)


===== 수집 경계 확인 =====
수집 종료 10분 이내에 끊긴 청평 episode: 7
수집 시작 경계 예외에 해당하는 episode: 8

[수집 종료 경계 예시]


,date,route_name,vehicle_id,cp_episode,cp_last_seen,cp_collection_end,minutes_before_collection_end
0,2026-09-14,1330-2,239000047,2026-09-14_청평터미널_1330-2_239000047_1,2026-09-14 20:58:47,2026-09-14 20:58:47,0.0
5,2026-09-14,1330-44,239000017,2026-09-14_청평터미널_1330-44_239000017_1,2026-09-14 20:58:47,2026-09-14 20:58:47,0.0
12,2026-09-15,1330-2,239000047,2026-09-15_청평터미널_1330-2_239000047_4,2026-09-15 20:58:47,2026-09-15 20:58:47,0.0
46,2026-09-15,1330-44,239000007,2026-09-15_청평터미널_1330-44_239000007_4,2026-09-15 20:58:47,2026-09-15 20:58:47,0.0
98,2026-09-16,1330-44,239000026,2026-09-16_청평터미널_1330-44_239000026_4,2026-09-16 20:58:47,2026-09-16 20:58:47,0.0
141,2026-09-17,1330-44,239000017,2026-09-17_청평터미널_1330-44_239000017_4,2026-09-17 20:58:47,2026-09-17 20:58:47,0.0
184,2026-09-18,1330-44,239000007,2026-09-18_청평터미널_1330-44_239000007_4,2026-09-18 20:58:48,2026-09-18 20:58:48,0.0



[수집 시작 경계 예시]


,date,route_name,vehicle_id,cp_episode,cp_first_seen,min_location_episode,min_eta_episode,matched_to_maseok
9,2026-09-15,1330-2,239000047,2026-09-15_청평터미널_1330-2_239000047_1,2026-09-15 06:02:47,2,122,True
43,2026-09-15,1330-44,239000007,2026-09-15_청평터미널_1330-44_239000007_1,2026-09-15 06:00:49,1,83,True
62,2026-09-16,1330-2,239000114,2026-09-16_청평터미널_1330-2_239000114_1,2026-09-16 06:00:48,2,155,True
95,2026-09-16,1330-44,239000026,2026-09-16_청평터미널_1330-44_239000026_1,2026-09-16 06:00:48,1,17,True
105,2026-09-17,1330-2,239000050,2026-09-17_청평터미널_1330-2_239000050_1,2026-09-17 06:00:49,3,175,True
138,2026-09-17,1330-44,239000017,2026-09-17_청평터미널_1330-44_239000017_1,2026-09-17 06:00:49,1,76,True
145,2026-09-18,1330-2,239000031,2026-09-18_청평터미널_1330-2_239000031_1,2026-09-18 06:02:47,3,183,True
181,2026-09-18,1330-44,239000007,2026-09-18_청평터미널_1330-44_239000007_1,2026-09-18 06:00:48,1,22,True


## 13. 최종 도착 Event 생성

청평 접근 흐름, 이후 마석역의 동일 운행 확인, 수집 시작·종료 경계를 함께 적용함. `arrival_time_est`는 정확한 실제 도착시각이 아니라 청평터미널 실시간 관측 기반 추정시각임.


In [16]:
# ============================================================
# 9. 접근 흐름 + 마석 연결 + 수집 경계 보정으로 최종 도착 event 생성
# ============================================================

def check_approach_flow(group):
    g = group.sort_values("recorded_at").copy()

    eta = pd.to_numeric(
        g["predict_time_sec"], errors="coerce"
    ).dropna()

    loc = pd.to_numeric(
        g["location_no"], errors="coerce"
    ).dropna()

    min_eta = eta.min() if len(eta) else np.nan
    min_location = loc.min() if len(loc) else np.nan

    eta_approached = (
        len(eta) >= 2
        and eta.iloc[-1] < eta.iloc[0]
    )

    location_approached = (
        len(loc) >= 2
        and loc.iloc[-1] < loc.iloc[0]
    )

    came_near = (
        (pd.notna(min_location) and min_location <= 3)
        or
        (pd.notna(min_eta) and min_eta <= 300)
    )

    approach_flow = (
        came_near
        and (eta_approached or location_approached)
    )

    return pd.Series({
        "min_eta_episode": min_eta,
        "min_location_episode": min_location,
        "approach_flow": approach_flow
    })


cp_flow = (
    df14[df14["station_name"] == "청평터미널"]
    .groupby(
        ["date", "route_name", "vehicle_id", "episode_id"]
    )
    .apply(check_approach_flow, include_groups=False)
    .reset_index()
    .rename(columns={"episode_id": "cp_episode"})
)

arrival = cp17.merge(
    cp_flow,
    on=["date", "route_name", "vehicle_id", "cp_episode"],
    how="left"
)

arrival = arrival.merge(
    matched17[
        ["cp_episode", "ms_episode", "ms_last_seen", "last_seen_travel_min"]
    ],
    on="cp_episode",
    how="left"
)

arrival["matched_to_maseok"] = arrival["ms_episode"].notna()

arrival["minutes_before_collection_end"] = (
    arrival["cp_collection_end"] - arrival["cp_last_seen"]
).dt.total_seconds() / 60

arrival["cut_by_collection_end"] = (
    arrival["minutes_before_collection_end"]
    .between(0, 10, inclusive="both")
)

# 수집 시작 직후 이미 정류장 가까이에 있던 버스를 위한 경계 예외
arrival["start_boundary_arrival"] = (
    (arrival["cp_first_seen"].dt.hour == 6)
    & (arrival["cp_first_seen"].dt.minute <= 2)
    & (arrival["min_location_episode"] <= 3)
    & (arrival["min_eta_episode"] <= 300)
    & arrival["matched_to_maseok"].fillna(False)
)

arrival["is_arrival_event"] = (
    (
        arrival["approach_flow"].fillna(False)
        | arrival["start_boundary_arrival"].fillna(False)
    )
    & arrival["matched_to_maseok"].fillna(False)
    & ~arrival["cut_by_collection_end"].fillna(False)
)

# 정확한 승강장 도착 순간이 아니라 마지막 API 관측시각
arrival["arrival_time_est"] = arrival["cp_last_seen"]

final_arrivals = (
    arrival[arrival["is_arrival_event"]]
    .copy()
    .sort_values(["date", "arrival_time_est"])
    .reset_index(drop=True)
)

print("===== 최종 도착 event 판정 =====")
print("청평 전체 episode:", len(arrival))
print("마석 정상 연결:", int(arrival["matched_to_maseok"].sum()))
print("접근 흐름 확인:", int(arrival["approach_flow"].sum()))
print("수집 종료로 제외:", int(arrival["cut_by_collection_end"].sum()))
print("최종 도착 event:", len(final_arrivals))


===== 최종 도착 event 판정 =====
청평 전체 episode: 191
마석 정상 연결: 183
접근 흐름 확인: 185
수집 종료로 제외: 7
최종 도착 event: 183
